# 07 - Supplementary analysis for the thesis

Needs the results of notebooks 02-05 on Google Drive. Nothing here trains a model or calls the Groq API.
It adds what the supervisor review asked for:

1. split sizes and class counts (train / validation / test),
2. a Bangla word-list baseline built from the training split only,
3. a common-subset comparison (the comments the LLM classified) next to the full-test comparison,
4. 95% bootstrap intervals and an exact McNemar test (BanglaBERT vs XLM-RoBERTa, first run),
5. the class balance of the comments the LLM did not classify,
6. inference timing details, learning-curve numbers, and error samples for the qualitative table.

Everything is written to `results/analysis/`.

In [ ]:
# Google Colab setup (does nothing when run locally).
# Clones the repo into Google Drive so results/ survive disconnects, then installs dependencies.
# Before the first run: Runtime > Change runtime type > T4 GPU, and upload train.csv (see below).
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/Set_A'
    if not os.path.isdir(REPO_DIR):
        !git clone -b main https://github.com/mehediparvez/bangla_profanity_word_detection.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}/BanglaProfanityDetection/notebooks
    !pip install -q transformers datasets accelerate scikit-learn better-profanity groq google-generativeai pyyaml seaborn tabulate
    if not os.path.exists('../data/train.csv'):
        print(f'Missing dataset: upload train.csv to {REPO_DIR}/BanglaProfanityDetection/data/train.csv')

In [ ]:
import os, sys, json, time
import numpy as np
import pandas as pd
sys.path.append('..')
from src.dataset import load_dataset
from src.evaluate import evaluate_model
from src.model import load_trained_model
from src.train import split_validation
from src.llm_baseline import _load_cache, _text_key
from src.lexicon import select_lexicon, tokenize, predict as lexicon_predict
from src.analysis import (subset_scores, bootstrap_ci, mcnemar_exact, coverage_check,
                          sample_errors, toxic_f1, weighted_f1)
from src.results import build_evaluation, save_evaluation
from src.utils import load_config, ensure_dirs

config = load_config('../configs/config.yaml')
OUT = '../results/analysis'
ensure_dirs(OUT)

## 1. Data splits

In [ ]:
seed = config['data']['random_seed']
train_texts, test_texts, train_labels, test_labels = load_dataset(
    '../' + config['data']['train_path'],
    text_column=config['data']['text_column'], label_column=config['data']['label_column'],
    test_size=config['data']['test_size'], random_seed=seed)
fit_texts, val_texts, fit_labels, val_labels = split_validation(
    train_texts, train_labels, val_size=config['training']['val_size'], seed=seed)

def dist(name, y):
    y = np.asarray(y)
    print(f'{name:<12} n={len(y):>6}  toxic={int(y.sum()):>5}  non-toxic={int((1 - y).sum()):>5}  toxic share={y.mean():.3f}')

splits = {}
for n, lab in [('fit', fit_labels), ('validation', val_labels), ('train total', train_labels), ('test', test_labels)]:
    dist(n, lab)
    splits[n] = {'n': len(lab), 'toxic': int(sum(lab))}

In [ ]:
# Content of the final dataset (Table 3.5 in the thesis): comments containing each kind of character
import re
all_texts = train_texts + test_texts
patterns = {
    'bangla_script': r'[\u0980-\u09FF]',
    'latin_letters': r'[A-Za-z]',
    'digits': r'[0-9\u09E6-\u09EF]',
    'emoji': r'[\U0001F300-\U0001FAFF\u2600-\u27BF]',
    'urls': r'https?://|www\.',
}
script_counts = {k: sum(bool(re.search(p, t)) for t in all_texts) for k, p in patterns.items()}
script_counts['total'] = len(all_texts)
print(script_counts)

## 2. Bangla word-list baseline

Built from the fit split only. The two cut-offs (minimum number of comments a word appears in, minimum share of those that are toxic) are chosen on the validation split.

In [ ]:
from collections import Counter
lexicon, settings, val_f1 = select_lexicon(fit_texts, fit_labels, val_texts, val_labels)
print('Chosen settings:', settings, ' validation weighted F1:', round(val_f1, 4), ' words in list:', len(lexicon))
t0 = time.time()
lex_pred = lexicon_predict(test_texts, lexicon)
lex_time = time.time() - t0
evaluation = build_evaluation(test_labels, lex_pred, lex_time,
                              extra={'settings': settings, 'lexicon_size': len(lexicon), 'validation_f1': val_f1})
print(json.dumps(evaluation['metrics'], indent=1))
save_evaluation(evaluation, '../results', 'bangla_lexicon')
# Most frequent list words, to inspect the list before describing it in the thesis
freq = Counter(w for t in fit_texts for w in set(tokenize(t)))
print(sorted(lexicon, key=lambda w: -freq[w])[:40])

## 3. Predictions of every approach on the test set

Fine-tuned models: first run (the saved model, the same one behind the confusion matrices in the report). LLM: read from the saved label cache, no API call.

In [ ]:
import torch
from better_profanity import profanity
profanity.load_censor_words()
bp_pred = [1 if profanity.contains_profanity(t) else 0 for t in test_texts]

preds, timing = {}, {}
for key, name in [('banglabert', 'bangla-bert-base'), ('xlmr', 'xlm-roberta-base')]:
    model, tok = load_trained_model(f'../results/{name}/run_1/best_model')
    model.to('cuda' if torch.cuda.is_available() else 'cpu')
    evaluate_model(model, tok, test_texts[:64], test_labels[:64])      # warm-up pass
    secs = []
    for _ in range(3):                                                # three timed passes over the full test set
        m, p = evaluate_model(model, tok, test_texts, test_labels, max_length=256, batch_size=32)
        secs.append(m['inference_time_seconds'])
    preds[key] = [int(x) for x in p]
    timing[key] = {'seconds_mean': float(np.mean(secs)), 'seconds_std': float(np.std(secs)), 'repetitions': 3,
                   'batch_size': 32, 'includes_tokenisation': True, 'includes_gpu_transfer': True}
    print(key, timing[key])
    del model

LLM_MODEL = config['llm']['groq']['model']
cache = _load_cache('../results/llm/predictions_cache.json', LLM_MODEL)
llm_raw = [cache.get(_text_key(t)) for t in test_texts]
llm_mask = np.array([p is not None for p in llm_raw])
preds['lexicon_english'], preds['lexicon_bangla'] = bp_pred, lex_pred
preds['llm'] = [int(p) if p is not None else 0 for p in llm_raw]
print('LLM classified', int(llm_mask.sum()), 'of', len(llm_mask), ' model:', LLM_MODEL)

## 4. Common-subset and full-test comparison

In [ ]:
y = np.array(test_labels)
names = {'lexicon_english': 'better_profanity (English)', 'lexicon_bangla': 'Bangla word list',
         'llm': 'LLM zero-shot', 'banglabert': 'BanglaBERT (run 1)', 'xlmr': 'XLM-RoBERTa (run 1)'}
rows = []
for k, n in names.items():
    p = np.array(preds[k])
    rows.append({'model': n, 'subset': 'common (LLM-classified)', **subset_scores(y, p, llm_mask)})
    if k != 'llm':
        rows.append({'model': n, 'subset': 'full test', **subset_scores(y, p, np.ones(len(y), bool))})
table = pd.DataFrame(rows).round(4)
print(table.to_string(index=False))
table.to_csv(f'{OUT}/comparison_tables.csv', index=False)

## 5. Confidence intervals and paired test

In [ ]:
ci = {}
for k, n in names.items():
    mask = llm_mask if k == 'llm' else np.ones(len(y), bool)
    p = np.array(preds[k])
    ci[k] = {m: bootstrap_ci(y[mask], p[mask], f) for m, f in [('weighted_f1', weighted_f1), ('toxic_f1', toxic_f1)]}
    print(n, {m: tuple(round(v, 4) for v in r) for m, r in ci[k].items()})
only_b, only_x, pval = mcnemar_exact(y, preds['banglabert'], preds['xlmr'])
print(f'McNemar (run 1): only BanglaBERT correct={only_b}, only XLM-R correct={only_x}, p={pval:.4g}')

## 6. Were the unclassified LLM comments a random subset?

In [ ]:
from scipy.stats import chi2_contingency
cov = coverage_check(y, llm_mask)
print(json.dumps(cov, indent=1))
if cov['unclassified']['n']:
    tbl = [[cov['classified']['toxic'], cov['classified']['n'] - cov['classified']['toxic']],
           [cov['unclassified']['toxic'], cov['unclassified']['n'] - cov['unclassified']['toxic']]]
    print('chi-square p-value for equal class balance:', chi2_contingency(tbl)[1])

## 7. Learning-curve numbers

In [ ]:
curve_rows = []
for key in ['banglabert', 'xlmr']:
    path = f'../results/{key}/learning_curve.json'
    if os.path.exists(path):
        for r in json.load(open(path)):
            curve_rows.append({'model': key, 'fraction': r['fraction'], 'train_comments': r.get('num_samples'),
                               'accuracy': r.get('eval_accuracy'), 'f1': r.get('eval_f1')})
curve = pd.DataFrame(curve_rows).round(4)
print(curve.to_string(index=False))
curve.to_csv(f'{OUT}/learning_curve_table.csv', index=False)

## 8. Error samples for the qualitative table

Random samples with a fixed seed. Read them, remove anything identifying (names, handles, links, phone numbers) and choose the ones to quote.

In [ ]:
errs = {}
for k in ['banglabert', 'xlmr', 'llm']:
    mask = llm_mask if k == 'llm' else np.ones(len(y), bool)
    idx = np.where(mask)[0]
    t = [test_texts[i] for i in idx]
    errs[k] = {et: sample_errors(t, y[idx], np.array(preds[k])[idx], et, k=10)
               for et in ['false_positive', 'false_negative']}
    for et, items in errs[k].items():
        print(f'--- {k} {et}')
        for e in items:
            print(' ', e['text'][:160])

## 9. Save everything

In [ ]:
with open(f'{OUT}/analysis.json', 'w', encoding='utf-8') as f:
    json.dump({'splits': splits, 'script_content': script_counts, 'lexicon': {'settings': settings, 'size': len(lexicon), 'validation_f1': val_f1},
               'timing': timing, 'ci': ci, 'mcnemar': {'only_banglabert': only_b, 'only_xlmr': only_x, 'p': pval},
               'coverage': cov, 'error_samples': errs}, f, ensure_ascii=False, indent=2)
pd.DataFrame({**{k: preds[k] for k in preds}, 'true': test_labels, 'llm_classified': llm_mask}).to_csv(
    f'{OUT}/test_predictions.csv', index=False)
print('Saved to', OUT)

In [ ]:
# Colab: make sure everything written to Google Drive is actually uploaded before the runtime ends
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.flush_and_unmount()
    print('All results saved to Google Drive.')